# Библиотеки

In [3]:
from itertools import combinations

import pandas as pd
import numpy as np
from scipy import stats
import pingouin as pg

import seaborn as sns

## 9.20 Таблицы сопряженности: МакНемар для категор. признаков в завис.

In [4]:
df = pg.read_dataset('cochran')

In [9]:
df.head(3)

,Subject,Time,Energetic
0,1,Monday,1
1,2,Monday,0
2,3,Monday,0


In [5]:
pg.cochran(data=df, dv='Energetic', within='Time', subject='Subject')

,Source,dof,Q,p-unc
cochran,Time,2,6.705882,0.034981


In [7]:
df_wide = df.pivot_table(index="Subject", columns="Time", values="Energetic")

In [10]:
df_wide.head(3)

Time,Friday,Monday,Wednesday
Subject,,,
1,1.0,1.0,1.0
2,1.0,0.0,1.0
3,1.0,0.0,0.0


In [8]:
pg.cochran(df_wide)

,Source,dof,Q,p-unc
cochran,Within,2,6.705882,0.034981


In [11]:
time = df['Time'].unique().tolist() # список уникальных значений Time

In [12]:
p = list(combinations(time,2)) # Комбинации пар - наши сравниваемые пары групп

In [13]:
results_list = [] # Создаем пустой список, который наполним результатами теста МакНемара под каждую пару из цикла - и потом в конце загоним его в датафрейм

In [14]:
for x, y in p: # Запуск цикла по всем парам столбцов, а в 'p' у нас уже есть список пар  в виде кортежей
    cross, res = pg.chi2_mcnemar(df.pivot_table(index='Subject', columns='Time', values='Energetic').dropna(), x=x, y=y) # поскольку тест МакНемара возвращает два DataFrame раскладываем их в две разные переменные, а в data даем датафрейм преобразованный в wide с удаленными строками с пустотами
    cp=cross.iloc[0, 1]+cross.iloc[1, 0] #Вычисляем суммы ячеек с изменениями 0->1 и 1->0 из каждого кросстаба
    results_list.append({
        'Сравниваемые периоды': f'{x}_vs_{y}',
        'К-во пар с изменениями':cp,
        'chi2': res['chi2'].iloc[0],
        'p_approx': res['p-approx'].iloc[0],
        'p_exact': res['p-exact'].iloc[0]
    }) # это словарик с ключами, в который мы забираем нужные столбцы из кажодго res и их значения - добавляя их в пустой список results_list

In [15]:
rd = pd.DataFrame(results_list) # записанные нужные нам результаты запихиваем в результирующий датафрейм

In [16]:
rd['pvals'] = rd.apply(lambda x: x['p_exact'] if x['К-во пар с изменениями'] < 25 else x['p_approx'], axis=1) # отбираем p-exact или p-approx по которым принимаются решения о значимости при сравнении 2-х групп

z, rd['p-corr'] = pg.multicomp(pvals=rd['pvals'].values, method='bonf')# поскольку результатом pg.multicomp() является два массива со списками, то отсекаем первый массив в z, а данные со второго записываем в столбец датафрейма

rd['p-corr2'] = (rd['pvals'] * rd['Сравниваемые периоды'].count()).clip(upper=1) # поскольку поправка Бонферрони по сути просто умножение p-val на количество сравниваемых пар, то вычислим ее еще простым способом, граничив уровень значимости 1 если вдруг "вылетим" после умножения за ее пределы

rd['Вывод'] = "" # Создаем столбец заполненный пусто

rd.loc[rd['p-corr'] <= 0.05, 'Вывод'] = "значимо" # Заполняем в нем строки соотв. условию  rd['p-corr'] <= 0.05 выводом "значимо"

In [17]:
rd

,Сравниваемые периоды,К-во пар с изменениями,chi2,p_approx,p_exact,pvals,p-corr,p-corr2,Вывод
0,Monday_vs_Wednesday,13,2.769231,0.096092,0.092285,0.092285,0.276855,0.276855,
1,Monday_vs_Friday,12,4.083333,0.043308,0.038574,0.038574,0.115723,0.115723,
2,Wednesday_vs_Friday,9,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,
